# 01 · Data collection

MotoGP-class results for every Grand Prix from **2013** (the first season of the Q1/Q2
qualifying format) to the latest completed round of **2026**. For each event we keep:

* **Q1 and Q2** classifications (to rebuild the starting grid and qualifying pace),
* the **Sprint** (introduced in 2023),
* the **Race**, plus the track/air temperature, humidity and wet/dry state for each session.

The source is the public results API behind motogp.com
(`api.motogp.pulselive.com/motogp/v1/results`). It is **unofficial and undocumented**, so
`motogp_analytics.data` is deliberately gentle. Every URL is fetched once and cached on disk
(`data/raw/`, git-ignored), requests are throttled to about 2 per second, and errors back off.
The tidy tables in `data/processed/` are committed.

In [1]:
import pandas as pd

from motogp_analytics.data import build_dataset

pd.set_option("display.max_columns", 30)
tables = build_dataset(start=2013, end=2026)
events, sessions, results = tables["events"], tables["sessions"], tables["results"]
{name: df.shape for name, df in tables.items()}

2013: 18 events, 872 result rows so far


2014: 18 events, 1745 result rows so far


2015: 18 events, 2683 result rows so far


2016: 18 events, 3495 result rows so far


2017: 18 events, 4342 result rows so far


2018: 19 events, 5296 result rows so far


... (5 outputs left out)

2024: 20 events, 11377 result rows so far


2025: 22 events, 12860 result rows so far


2026: 20 events, 14187 result rows so far


{'events': (262, 12), 'sessions': (871, 11), 'results': (14187, 23)}

In [2]:
coverage = sessions.groupby(["season", "session"]).size().unstack(fill_value=0)
coverage.insert(0, "events", events.groupby("season").size())
coverage

session,events,Q1,Q2,RAC,SPR
season,,,,,
2013,18,17,17,18,0
2014,18,18,18,18,0
2015,18,18,18,18,0
2016,18,18,18,19,0
2017,18,18,18,18,0
2018,19,19,19,20,0
2019,19,19,19,19,0
2020,14,14,14,16,0
2021,18,18,18,18,0


## Quirks worth knowing

**Restarted races.** When a race is red-flagged and restarted, the API returns two `RAC`
sessions for the event. The later one is the official result, and `features.official_sessions`
keeps only that one.

In [3]:
rac = sessions[sessions["session"] == "RAC"]
multi = rac.groupby("event_id").size().loc[lambda s: s > 1].index
events[events["event_id"].isin(multi)][["season", "round", "event_name", "circuit_name"]]

,season,round,event_name,circuit_name
61,2016,8,TT ASSEN,TT Circuit Assen
108,2018,19,GRAN PREMIO DE LA COMUNITAT VALENCIANA,Circuit Ricardo Tormo
131,2020,5,MOTORRAD GRAND PRIX VON ÖSTERREICH,Red Bull Ring - Spielberg
132,2020,6,GRAND PRIX OF STYRIA,Red Bull Ring - Spielberg
247,2026,6,GRAND PRIX OF CATALONIA,Circuit de Barcelona-Catalunya


**Events without a race classification.** Cancelled or abandoned races still appear as events.
These are dropped from race-level analysis.

In [4]:
race_results = results[results["session"] == "RAC"]
no_race = events[~events["event_id"].isin(race_results["event_id"])]
no_race[["season", "round", "event_name", "date_start"]]

,season,round,event_name,date_start
101,2018,12,BRITISH GRAND PRIX,2018-08-24
259,2026,18,GRAND PRIX OF AUSTRALIA,2026-10-23


**Classification status codes.** `INSTND` = classified, `OUTSTND` = did not finish. The rest
are rarer: not on the restart grid, did not finish the first part of a restarted race,
disqualified, and so on.

In [5]:
results.loc[results["session"] == "RAC", "status"].value_counts()

status
INSTND              4822
OUTSTND             1022
NOTFINISHFIRST        96
NOTONRESTARTGRID      14
NOTSTARTED             8
DISQUALIFIED           6
FINISHEDTHRUPITS       2
EXCLUDED               1
Name: count, dtype: int64

In [6]:
results.head()

,season,round,event_id,session,session_id,session_date,position,rider_id,rider_name,rider_number,rider_country,team_name,constructor,points,status,total_laps,time,time_s,best_lap_s,top_speed,average_speed,gap_first_s,gap_laps
0,2013,1,e5c416e6-40f1-43f4-85a1-ae0486fb59e1,Q1,5ffe7243-c0c4-4419-96d8-aead0bc3058b,2013-04-06T20:35:00+00:00,1.0,a2f51450-cb43-4d32-8eef-bda9ebb435ed,Aleix Espargaro,41,ES,Power Electronics Aspar,ART,0.0,INSTND,4,None,NaN,117.151,315.4,NaN,0.000,NaN
1,2013,1,e5c416e6-40f1-43f4-85a1-ae0486fb59e1,Q1,5ffe7243-c0c4-4419-96d8-aead0bc3058b,2013-04-06T20:35:00+00:00,2.0,8c0dc3c9-48bc-4e28-95c7-5671171ed7a6,Andrea Iannone,29,IT,Energy T.I. Pramac Racing,Ducati,0.0,INSTND,6,None,NaN,117.203,339.0,NaN,0.052,NaN
2,2013,1,e5c416e6-40f1-43f4-85a1-ae0486fb59e1,Q1,5ffe7243-c0c4-4419-96d8-aead0bc3058b,2013-04-06T20:35:00+00:00,3.0,d6e07f2a-a327-40ab-874b-e957dcc74705,Ben Spies,11,US,Ignite Pramac Racing,Ducati,0.0,INSTND,6,None,NaN,117.440,334.7,NaN,0.289,NaN
3,2013,1,e5c416e6-40f1-43f4-85a1-ae0486fb59e1,Q1,5ffe7243-c0c4-4419-96d8-aead0bc3058b,2013-04-06T20:35:00+00:00,4.0,396b664f-b3ac-4e16-9612-21150f220e85,Randy De Puniet,14,FR,Power Electronics Aspar,ART,0.0,INSTND,7,None,NaN,117.551,316.4,NaN,0.400,NaN
4,2013,1,e5c416e6-40f1-43f4-85a1-ae0486fb59e1,Q1,5ffe7243-c0c4-4419-96d8-aead0bc3058b,2013-04-06T20:35:00+00:00,5.0,e0ed2e60-b2b8-4be9-bbb3-608feb8d08d8,Lukas Pesek,52,CZ,Came IodaRacing Project,Ioda-Suter,0.0,INSTND,7,None,NaN,117.926,314.5,NaN,0.775,NaN
